In [40]:
# importando bibliotecas
from faker import Faker
from random import choice, choices, randint, triangular, uniform
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from unicodedata import normalize
from re import sub, IGNORECASE

In [41]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o banco de dados
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
faker = Faker('pt_BR')

In [42]:
def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def remove_acentos(texto: str) -> str:
    return normalize('NFKD', texto.lower()).encode('ASCII', 'ignore').decode('ASCII')

In [43]:
# Constantes
QTD_LOJAS               = 10
QTD_BANCOS              = 5
QTD_CONTAS              = 120
QTD_PAGAMENTOS          = 1000
QTD_VENDAS              = 10000
QTD_TRANSPORTADORAS     = 8
QTD_FUNCIONARIOS        = 50
QTD_CONTA_LOJA          = 15
QTD_CONTA_FUNCIONARIO   = 60
QTD_PAGAMENTO_PARCELADO = 300

In [44]:
dados_loja = []
for i in range(QTD_LOJAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_loja.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'email': faker.email(),
        'telefone': telefone_formatado,
        'percentual_comissao': round(uniform(0.01, 0.2), 2)
    })

df_lojas = pd.DataFrame(dados_loja)
display(df_lojas.head())

,nome,cnpj,email,telefone,percentual_comissao
0,Moraes,71308652000136,maria-juliafernandes@example.com,5551968984059,0.12
1,Sampaio e Filhos,81375296000127,jose98@example.com,5595998034669,0.07
2,Azevedo Sá S/A,71294386000130,nda-costa@example.com,5587943045784,0.11
3,Costa Ltda.,40976823000171,ravy16@example.net,5535969621565,0.03
4,Sampaio,98304156000160,zbarbosa@example.net,5531934040049,0.04


In [45]:
dados_banco = []
for i in range(QTD_BANCOS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    
    dados_banco.append({
        'nome': f'Banco {faker.company()}',
        'cnpj': cnpj_formatado
    })

df_bancos = pd.DataFrame(dados_banco)
display(df_bancos.head())

,nome,cnpj
0,Banco Brito,32154680000194
1,Banco Sampaio - EI,16350427000150
2,Banco Jesus Moura - EI,98013624000148
3,Banco da Paz,93806724000108
4,Banco Rezende Nunes - EI,32671804000109


In [46]:
dados_conta = []

TIPOS_CONTA = ['CORRENTE', 'POUPANCA', 'SALARIO']

for i in range(QTD_CONTAS):
    dados_conta.append({
        'agencia': str(randint(1, 9999)).zfill(4),
        'numero_conta': str(randint(1, 999999)).zfill(6),
        'tipo_conta': choice(TIPOS_CONTA),
        'saldo': round(triangular(0, 9999.99, 100), 2),
        'cod_banco': randint(1, QTD_BANCOS)
    })

df_contas = pd.DataFrame(dados_conta)
display(df_contas.head())

,agencia,numero_conta,tipo_conta,saldo,cod_banco
0,1698,823851,SALARIO,3025.74,3
1,8111,560577,SALARIO,7677.60,5
2,6611,614085,SALARIO,3603.32,1
3,3399,366488,POUPANCA,1761.74,5
4,3379,408817,SALARIO,239.86,2


In [47]:
dados_pagamento = []

TIPOS_PAGAMENTO = ['BOLETO', 'PIX', 'DEBITO', 'CREDITO']
STATUS_PAGAMENTO = ['AGUARDANDO', 'CONCLUIDO', 'EXPIRADO']

for i in range(QTD_PAGAMENTOS):
    cod_conta_origem = randint(1, QTD_CONTAS)
    cod_conta_destino = randint(1, QTD_CONTAS)
    while cod_conta_destino == cod_conta_origem:
        cod_conta_destino = randint(1, QTD_CONTAS)
    
    dados_pagamento.append({
        'tipo_pagamento': choice(TIPOS_PAGAMENTO),
        'status_pagamento': choice(STATUS_PAGAMENTO),
        'valor': round(triangular(0, 999.99, 100), 2),
        'cod_conta_origem': cod_conta_origem,
        'cod_conta_destino': cod_conta_destino
    })

df_pagamentos = pd.DataFrame(dados_pagamento)
display(df_pagamentos.head())

,tipo_pagamento,status_pagamento,valor,cod_conta_origem,cod_conta_destino
0,BOLETO,AGUARDANDO,180.23,96,92
1,PIX,AGUARDANDO,283.96,78,67
2,DEBITO,AGUARDANDO,391.18,115,66
3,CREDITO,CONCLUIDO,92.80,83,89
4,PIX,CONCLUIDO,38.23,5,67


In [48]:
dados_funcionario = []
for i in range(QTD_FUNCIONARIOS):
    nome = sub(r'\b(sr|sra|srta|dr|dra)\.\s*', '', faker.name(), flags=IGNORECASE)
    cpf_formatado = apenas_numeros(faker.cpf())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    email = f'{sub(r' ', '_', remove_acentos(nome))}@gmail.com'
    
    dados_funcionario.append({
        'nome': nome,
        'cpf': cpf_formatado,
        'cargo': 'Vendedor',
        'salario': round(triangular(1621, 4000, 100), 2),
        'email': email,
        'telefone': telefone_formatado,
        'cod_loja': randint(1, QTD_LOJAS)
    })

df_funcionarios = pd.DataFrame(dados_funcionario)
display(df_funcionarios.head())

,nome,cpf,cargo,salario,email,telefone,cod_loja
0,Alana Rodrigues,98156430298,Vendedor,1982.72,alana_rodrigues@gmail.com,5512968333470,3
1,Leonardo Montenegro,72514309832,Vendedor,3539.65,leonardo_montenegro@gmail.com,5594991711328,7
2,Ana Laura Araújo,69807341213,Vendedor,1170.41,ana_laura_araujo@gmail.com,5522944158419,6
3,Luiza Rezende,01789465249,Vendedor,1566.11,luiza_rezende@gmail.com,5576950765266,7
4,Camila Ferreira,84069273565,Vendedor,2758.07,camila_ferreira@gmail.com,5545976982147,6


In [49]:
dados_venda = []

STATUS_VENDA = ['APROVADO', 'CANCELADO', 'ESTORNADO']

for i in range(QTD_VENDAS):
    cod_vendedor = randint(1, QTD_FUNCIONARIOS)
    cod_loja = df_funcionarios.loc[
        cod_vendedor - 1, 'cod_loja'
    ]
    
    dados_venda.append({
        'valor_total': round(triangular(0, 999.99, 100), 2),
        'status_venda': choices(STATUS_VENDA, weights=[85, 5, 10], k=1)[0],
        'cod_vendedor': cod_vendedor,
        'cod_loja': cod_loja
    })

df_vendas = pd.DataFrame(dados_venda)
display(df_vendas.head())

,valor_total,status_venda,cod_vendedor,cod_loja
0,231.79,APROVADO,31,1
1,826.54,APROVADO,9,3
2,98.15,ESTORNADO,3,6
3,233.03,APROVADO,45,1
4,280.42,APROVADO,18,2


In [50]:
dados_transportadora = []
for i in range(QTD_TRANSPORTADORAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_transportadora.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'telefone': telefone_formatado,
        'email': faker.email(),
        'taxa_entrega': round(uniform(0.01, 1), 2)
    })

df_transportadoras = pd.DataFrame(dados_transportadora)
display(df_transportadoras.head())

,nome,cnpj,telefone,email,taxa_entrega
0,Aragão,18952073000149,5582996262152,novaesyasmin@example.org,0.85
1,Silveira da Rosa - ME,62453807000137,5564956750075,aurora02@example.com,0.53
2,Fonseca,32648710000119,5554908536808,liviasilva@example.org,0.81
3,Nascimento,21580946000144,5561973816988,nabreu@example.org,0.98
4,Novais Novais - EI,35081762000144,5583914122757,bryanfernandes@example.com,0.51


In [51]:
dados_envia_itens = []
dados_conta_loja = []
dados_conta_funcionario = []
dados_pagamento_parcelado = []

dados_conta_loja = [{'cod_loja': i+1, 'cod_conta': randint(1, QTD_CONTAS)} for i in range(QTD_LOJAS)]
for i in range(QTD_CONTA_LOJA - QTD_LOJAS):
    dados_conta_loja.append({'cod_loja': randint(1, QTD_LOJAS), 'cod_conta': randint(1, QTD_CONTAS)})

dados_conta_funcionario = [{'cod_funcionario': i+1, 'cod_conta': randint(1, QTD_CONTAS)} for i in range(QTD_FUNCIONARIOS)]
for i in range(QTD_CONTA_FUNCIONARIO - QTD_FUNCIONARIOS):
    dados_conta_funcionario.append({'cod_funcionario': randint(1, QTD_FUNCIONARIOS), 'cod_conta': randint(1, QTD_CONTAS)})

dados_envia_itens = [
    {'cod_loja': l, 'cod_transportadora': t}
    for l in range(1, QTD_LOJAS + 1)
    for t in range(1, QTD_TRANSPORTADORAS + 1)
]

for i in range(QTD_PAGAMENTO_PARCELADO):
    dados_pagamento_parcelado.append({
        'cod_pagamento': randint(1, QTD_PAGAMENTOS),
        'cod_venda': randint(1, QTD_VENDAS)
    })

df_envia_itens = pd.DataFrame(dados_envia_itens)
df_conta_loja = pd.DataFrame(dados_conta_loja)
df_conta_funcionario = pd.DataFrame(dados_conta_funcionario)
df_pagamento_parcelado = pd.DataFrame(dados_pagamento_parcelado)

In [52]:
# Inserindo no banco de dados
tabelas = {
    'tb_loja': df_lojas,
    'tb_banco': df_bancos,
    'tb_transportadora': df_transportadoras,
    'tb_funcionario': df_funcionarios,
    'tb_conta': df_contas,
    'tb_pagamento': df_pagamentos,
    'tb_venda': df_vendas,
    'tb_envia_itens': df_envia_itens,
    'tb_conta_loja': df_conta_loja,
    'tb_conta_funcionario': df_conta_funcionario,
    'tb_pagamento_parcelado': df_pagamento_parcelado
}

print('Iniciando conexão com o banco de dados')
for t, df in tabelas.items():
    try:
        df.to_sql(name=t, con=engine, if_exists='append', index=False)
    except Exception as e:
        print(f'Um erro no banco aconteceu')
        print(e)
print('Conexão feita com sucesso')

Iniciando conexão com o banco de dados
Conexão feita com sucesso
